# 14b — Evaluation: 5,000-user Hyperbolic RBM (Rating Personality)

Two tracks:

1. **Real part** — held-out test rating prediction (Real RBM baseline = RBMB1 vs Hyperbolic joint)
2. **Imaginary part** — personality capture (t-SNE of h₂ colored by μ_user)

| Part | Section |
|------|--------|
| Part 0 | Setup / load |
| Part 1 | Real RBM baseline (W1 / bh1) |
| Part 1.5 | Mixed-sign occupancy |
| Part 2 | Hyperbolic joint prediction |
| Part 3 | Extreme-user group ablation |
| Part 4 | t-SNE of personality hidden activations |
| Part 5 | Summary table + CSV |

**Prerequisites:** notebooks 09b, 12b, 13b.


## Part 0 — Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

proc = root / "data" / "processed"
out_dir = root / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

RATING_LEVELS = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0], dtype=np.float64)
K = len(RATING_LEVELS)


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -60.0, 60.0)))


paths = {
    "W1": proc / "rbmB1_weights_5k.npy",
    "bh1": proc / "rbmB1_bias_hidden_5k.npy",
    "W2": proc / "rbmB2_weights_5k.npy",
    "bh2": proc / "rbmB2_bias_hidden_5k.npy",
    "B1": proc / "channelB1_softmax.npy",
    "B2": proc / "channelB2_personality.npy",
    "mask": proc / "mask.npy",
    "cohort": proc / "cohort_user_ids.npy",
    "vocab": proc / "movie_vocab.npy",
    "means": proc / "user_means.npy",
    "test": proc / "test_labels.csv",
}
for p in paths.values():
    assert p.exists(), f"Missing {p}"

W1 = np.load(paths["W1"])
bh1 = np.load(paths["bh1"])
W2 = np.load(paths["W2"])
bh2 = np.load(paths["bh2"])

channelB1 = np.load(paths["B1"], mmap_mode="r")
channelB2 = np.load(paths["B2"], mmap_mode="r")
mask = np.load(paths["mask"], mmap_mode="r")

cohort_user_ids = np.load(paths["cohort"]).astype(int)
movie_vocab = np.load(paths["vocab"]).astype(int)
user_means = np.load(paths["means"]).astype(np.float64)
test_labels = pd.read_csv(paths["test"])

n_users, n_movies, k_b1 = channelB1.shape
assert k_b1 == K
assert channelB2.shape == (n_users, n_movies, 1)
assert mask.shape == (n_users, n_movies)
assert W1.shape == (n_movies * K, bh1.shape[0])
assert W2.shape == (n_movies, bh2.shape[0])

user_to_row = {int(uid): i for i, uid in enumerate(cohort_user_ids)}
movie_to_col = {int(mid): j for j, mid in enumerate(movie_vocab)}

mu_global = float(np.mean(user_means))

print(f"Project root: {root}")
print(f"W1 {W1.shape}  bh1 {bh1.shape}")
print(f"W2 {W2.shape}  bh2 {bh2.shape}")
print(f"channelB1 {channelB1.shape}  channelB2 {channelB2.shape}  mask {mask.shape}")
print(f"cohort {cohort_user_ids.shape}  vocab {movie_vocab.shape}  user_means {user_means.shape}")
print(f"test_labels rows: {len(test_labels):,}")
print(f"mu_global (mean of user_means): {mu_global:.6f}")

## Part 1 — Real RBM baseline (W1 / bh1)

Real RBM ≡ RBMB1 from notebook 13b — no retrain.  
Predicted rating = expected value under reconstructed K-bin softmax for the test movie.

In [ ]:
print("Building X1 / X2 and hidden activations (one forward pass per user) …")
X1 = np.asarray(channelB1, dtype=np.float32).reshape(n_users, -1)
X2 = np.asarray(channelB2, dtype=np.float32).reshape(n_users, -1)

H1 = sigmoid(X1 @ W1 + bh1)  # (n_users, n_hidden)
H2 = sigmoid(X2 @ W2 + bh2)
print(f"H1 {H1.shape}  H2 {H2.shape}")

# Map test rows → (i, j); drop any ids outside cohort/vocab
test_df = test_labels.copy()
test_df["i"] = test_df["userId"].map(user_to_row)
test_df["j"] = test_df["movieId"].map(movie_to_col)
n_before = len(test_df)
test_df = test_df.dropna(subset=["i", "j"]).copy()
test_df["i"] = test_df["i"].astype(int)
test_df["j"] = test_df["j"].astype(int)
print(f"test rows usable: {len(test_df):,} / {n_before:,}")


def predict_r1_batch(i_arr, j_arr):
    """Softmax expected rating from B1 reconstruction for pairs (i, j)."""
    preds = np.empty(len(i_arr), dtype=np.float64)
    # Group by movie column for vectorized W-slice matmul
    order = np.argsort(j_arr)
    j_sorted = j_arr[order]
    i_sorted = i_arr[order]
    start = 0
    n = len(j_sorted)
    while start < n:
        j = int(j_sorted[start])
        end = start + 1
        while end < n and int(j_sorted[end]) == j:
            end += 1
        ii = i_sorted[start:end]
        W_slice = W1[j * K : (j + 1) * K]  # (K, n_hidden)
        logits = H1[ii] @ W_slice.T          # (batch, K)
        probs = sigmoid(logits)
        denom = probs.sum(axis=1)
        denom = np.where(denom > 0, denom, 1.0)
        r = (probs * RATING_LEVELS).sum(axis=1) / denom
        preds[order[start:end]] = r
        start = end
    return preds


i_all = test_df["i"].to_numpy()
j_all = test_df["j"].to_numpy()
y_true = test_df["rating"].to_numpy(dtype=np.float64)

print("Predicting Real RBM (r1) …")
r1 = predict_r1_batch(i_all, j_all)

rmse_real = float(np.sqrt(np.mean((r1 - y_true) ** 2)))
mae_real = float(np.mean(np.abs(r1 - y_true)))
print(f"Real RBM baseline (B1 only):  RMSE={rmse_real:.4f}  MAE={mae_real:.4f}  n={len(y_true):,}")

## Part 1.5 — Mixed-sign occupancy

Pre-sigmoid dual-channel hidden fields:
\(z_1 = X_1 W_1 + b_{h1}\), \(z_2 = X_2 W_2 + b_{h2}\).

A hidden unit is **mixed-sign** for a user when \(\mathrm{sign}(z_1) \neq \mathrm{sign}(z_2)\).
Report the fraction of mixed-sign \((\mathrm{user}, \mathrm{hidden})\) pairs.


In [ ]:
z1 = X1 @ W1 + bh1
z2 = X2 @ W2 + bh2
assert z1.shape == z2.shape == (n_users, W1.shape[1])

mixed = np.sign(z1) != np.sign(z2)
mixed_sign_rate = float(mixed.mean())
print(f"Mixed-sign occupancy (Rating Personality): {100.0 * mixed_sign_rate:.4f}%")


## Part 2 — Hyperbolic joint prediction

- **r1**: rating-channel reconstruction (Part 1)
- **v_recon2[j]**: personality-channel reconstruction at movie \(j\) (scalar residual)
- **r_joint** = clip(\(\mathrm{r}_1 + v_{\mathrm{recon}2}[j]\), 0.5, 5.0)

In [ ]:
def predict_v2_batch(i_arr, j_arr):
    """Personality-channel reconstructed scalar at movie j for user i."""
    preds = np.empty(len(i_arr), dtype=np.float64)
    order = np.argsort(j_arr)
    j_sorted = j_arr[order]
    i_sorted = i_arr[order]
    start = 0
    n = len(j_sorted)
    while start < n:
        j = int(j_sorted[start])
        end = start + 1
        while end < n and int(j_sorted[end]) == j:
            end += 1
        ii = i_sorted[start:end]
        # v_recon2[i,j] = 0.5 * tanh(H2[i] · W2[j])
        logits = H2[ii] @ W2[j]  # (batch,)
        preds[order[start:end]] = 0.5 * np.tanh(logits)
        start = end
    return preds


print("Predicting personality reconstr. (v_recon2) …")
v2 = predict_v2_batch(i_all, j_all)
r_joint = np.clip(r1 + v2, 0.5, 5.0)

rmse_hyp = float(np.sqrt(np.mean((r_joint - y_true) ** 2)))
mae_hyp = float(np.mean(np.abs(r_joint - y_true)))

compare = pd.DataFrame(
    [
        {"model": "Real RBM (B1 only)", "RMSE": rmse_real, "MAE": mae_real},
        {"model": "Hyperbolic joint", "RMSE": rmse_hyp, "MAE": mae_hyp},
    ]
)
print("\n=== Real-part comparison ===")
print(compare.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"ΔRMSE (Hyp − Real): {rmse_hyp - rmse_real:+.4f}")
print(f"ΔMAE  (Hyp − Real): {mae_hyp - mae_real:+.4f}")

test_df["r1"] = r1
test_df["v2"] = v2
test_df["r_joint"] = r_joint
test_df["y_true"] = y_true
test_df["mu_user"] = user_means[i_all]

## Part 3 — Extreme-user group ablation

- **generous:** μ_user > 4.0  
- **strict:** μ_user < 2.5  
- **middle:** everyone else

In [ ]:
def group_label(mu):
    if mu > 4.0:
        return "generous"
    if mu < 2.5:
        return "strict"
    return "middle"


user_group = np.array([group_label(mu) for mu in user_means])
test_df["group"] = user_group[i_all]

n_users_by_group = {
    g: int(np.sum(user_group == g)) for g in ("generous", "strict", "middle")
}
n_users_by_group["all"] = n_users

rows = []
for g in ("generous", "strict", "middle", "all"):
    if g == "all":
        sub = test_df
    else:
        sub = test_df[test_df["group"] == g]
    if len(sub) == 0:
        rows.append(
            {
                "group": g,
                "n_users": n_users_by_group[g],
                "n_test_ratings": 0,
                "Real_RMSE": np.nan,
                "Hyp_RMSE": np.nan,
                "Real_MAE": np.nan,
                "Hyp_MAE": np.nan,
            }
        )
        continue
    yt = sub["y_true"].to_numpy()
    pr = sub["r1"].to_numpy()
    ph = sub["r_joint"].to_numpy()
    rows.append(
        {
            "group": g,
            "n_users": n_users_by_group[g],
            "n_test_ratings": len(sub),
            "Real_RMSE": float(np.sqrt(np.mean((pr - yt) ** 2))),
            "Hyp_RMSE": float(np.sqrt(np.mean((ph - yt) ** 2))),
            "Real_MAE": float(np.mean(np.abs(pr - yt))),
            "Hyp_MAE": float(np.mean(np.abs(ph - yt))),
        }
    )

group_df = pd.DataFrame(rows)
group_df["delta_RMSE"] = group_df["Hyp_RMSE"] - group_df["Real_RMSE"]
print("=== Group ablation (negative delta_RMSE ⇒ Hyperbolic better) ===")
print(group_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

## Part 4 — Imaginary part: t-SNE of personality hidden activations

\(h_2[i] = \sigma(X_2[i] W_2 + b_{h2})\). Scatter colored by μ_user; generous / strict edged distinctly.

In [ ]:
from sklearn.manifold import TSNE

print("Running t-SNE on H2 (5000 × 128), perplexity=30 …")
tsne = TSNE(n_components=2, perplexity=30, random_state=42, init="pca", learning_rate="auto")
coords = tsne.fit_transform(H2.astype(np.float64))

generous_mask = user_means > 4.0
strict_mask = user_means < 2.5
middle_mask = ~(generous_mask | strict_mask)

fig, ax = plt.subplots(figsize=(8, 6.5))
sc = ax.scatter(
    coords[middle_mask, 0],
    coords[middle_mask, 1],
    c=user_means[middle_mask],
    cmap="coolwarm",
    s=8,
    alpha=0.55,
    vmin=user_means.min(),
    vmax=user_means.max(),
    linewidths=0,
    label="middle",
)
ax.scatter(
    coords[generous_mask, 0],
    coords[generous_mask, 1],
    c=user_means[generous_mask],
    cmap="coolwarm",
    s=28,
    alpha=0.9,
    vmin=user_means.min(),
    vmax=user_means.max(),
    edgecolors="darkred",
    linewidths=0.8,
    label=f"generous (n={int(generous_mask.sum())})",
)
ax.scatter(
    coords[strict_mask, 0],
    coords[strict_mask, 1],
    c=user_means[strict_mask],
    cmap="coolwarm",
    s=28,
    alpha=0.9,
    vmin=user_means.min(),
    vmax=user_means.max(),
    edgecolors="navy",
    linewidths=0.8,
    label=f"strict (n={int(strict_mask.sum())})",
)
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("μ_user (train mean rating)")
ax.set_title("t-SNE of Rating Personality hidden activations (H2)")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.legend(loc="best", fontsize=9)
fig.tight_layout()

path_tsne = out_dir / "tsne_personality_5k.png"
fig.savefig(path_tsne, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {path_tsne}")
print("Visual clustering: inspect the figure (generous=darkred edge, strict=navy edge).")

## Part 5 — Summary table

In [ ]:
print("Model               RMSE    MAE")
print(f"Real RBM (B1 only)  {rmse_real:.3f}   {mae_real:.3f}")
print(f"Hyperbolic joint    {rmse_hyp:.3f}   {mae_hyp:.3f}")
print()
print(f"{'Group':<10} {'n_users':>8} {'Real_RMSE':>10} {'Hyp_RMSE':>10} {'delta':>8}")
for _, row in group_df.iterrows():
    if row["group"] == "all":
        continue
    print(
        f"{row['group']:<10} {int(row['n_users']):>8} "
        f"{row['Real_RMSE']:>10.3f} {row['Hyp_RMSE']:>10.3f} {row['delta_RMSE']:>+8.3f}"
    )

summary_rows = [
    {"section": "model", "name": "Real RBM (B1 only)", "RMSE": rmse_real, "MAE": mae_real},
    {"section": "model", "name": "Hyperbolic joint", "RMSE": rmse_hyp, "MAE": mae_hyp},
]
for _, row in group_df.iterrows():
    if row["group"] == "all":
        continue
    summary_rows.append(
        {
            "section": "group",
            "name": row["group"],
            "n_users": int(row["n_users"]),
            "n_test_ratings": int(row["n_test_ratings"]),
            "Real_RMSE": row["Real_RMSE"],
            "Hyp_RMSE": row["Hyp_RMSE"],
            "delta": row["delta_RMSE"],
            "Real_MAE": row["Real_MAE"],
            "Hyp_MAE": row["Hyp_MAE"],
        }
    )

summary_df = pd.DataFrame(summary_rows)
path_csv = out_dir / "evaluation_summary_5k.csv"
summary_df.to_csv(path_csv, index=False)
print(f"\nSaved: {path_csv}")
print(f"Saved: {out_dir / 'tsne_personality_5k.png'}")